# Full benchmark run — dev (60) + locked eval (240)

Run `00_colab_smoke.ipynb` successfully first.

**Runtime: choose A100 if available** (執行階段 → 變更執行階段類型 → A100).
This is the one lever that meaningfully shortens the run; batching the scoring passes
would be faster but is deliberately not done — it would break the batch-1 determinism
promise in MODEL_CARD.md and make these numbers incomparable to the smoke run.

**Time budget — derived from the REAL smoke run (T4, 2026-07-24), not guesses.**
Measured per-sample attribution cost: mode A ≈ 122 s + mode B ≈ 75 s + generation ≈ 8 s
≈ **3.4 min/sample on T4**.

| section | samples | T4 (measured basis) | A100 (est. 3–4x faster) |
|---|---|---|---|
| dev | 60 | ~3.5 h | ~1 h |
| locked eval | 240 | ~13–14 h | ~3.5–4.5 h |
| **total** | 300 | **~17 h** | **~5 h** |

On T4 the eval section will NOT finish in one sitting — multi-session resume is the
designed path, not a failure. On A100 it is realistically one long session.

**Disconnect protocol:** checkpoints live on Drive with per-sample granularity. If the
runtime dies: Reconnect → Runtime → Run all. Completed samples are skipped; at most the
in-flight sample is lost. A changed config refuses to resume (new run required) — that is
intentional.

**Optional: two parallel sessions.** The dev and eval sections write to disjoint paths
(`results/raw/dev/…` vs `results/raw/eval/…`, and split-scoped caches), so you may run
them in two Colab sessions at once. The only shared file is `results/derived/summary.json`
(both `evaluate` cells rewrite it, last-writer-wins) — harmless, because it is fully
regenerable from the raw records locally after import.

**Not included here:** `contextcite` (measured 81 s/sample → would add ~6.7 h on T4 for
300 samples; already characterised on the smoke split) and the experimental `arc_jsd`.

**Lock discipline:** the eval split is locked. It is run once per benchmark version;
justify any re-run in the run manifest notes.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU runtime — Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_RESULTS = '/content/drive/MyDrive/reab/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.environ['RAG_EVIDENCE_RESULTS_RAW'] = f'{DRIVE_RESULTS}/raw'
os.environ['RAG_EVIDENCE_RESULTS_DERIVED'] = f'{DRIVE_RESULTS}/derived'
os.environ['HF_HOME'] = '/content/hf_cache'

# No manual Drive folder for the bundle: pick the file from your computer.
BUNDLE_ZIP = '/content/reab_bundle.zip'
if not os.path.exists(BUNDLE_ZIP):
    from google.colab import files

    print('Select reab_bundle.zip from your computer:')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file selected — re-run this cell and choose reab_bundle.zip.')
    picked = next(iter(uploaded))
    if not picked.lower().endswith('.zip'):
        raise SystemExit(f'{picked!r} is not a .zip — re-run this cell and choose reab_bundle.zip.')
    os.replace(picked, BUNDLE_ZIP)

print(f'bundle ready: {BUNDLE_ZIP} ({os.path.getsize(BUNDLE_ZIP) / 1e6:.1f} MB)')

In [ ]:
!rm -rf /content/reab && mkdir -p /content/reab
!unzip -q "$BUNDLE_ZIP" -d /content/reab
%cd /content/reab
!pip install -q "transformers==5.14.1" "tokenizers==0.22.2" "accelerate>=1.0" "bitsandbytes>=0.49" "datasets>=3.0" "sentence-transformers>=5.0"
!pip install -q -e ".[ml,gpu]"
!mkdir -p "$DRIVE_RESULTS" && cp -rn results/raw "$DRIVE_RESULTS/" 2>/dev/null; cp -rn results/derived "$DRIVE_RESULTS/" 2>/dev/null; true

## Section 1 — dev split (60 questions)

In [ ]:
!python -m rag_evidence.cli status --config configs/dev.yaml

In [ ]:
# dense retrieval for dev/eval runs here (GPU-fast); bm25 came with the bundle
!python -m rag_evidence.cli retrieve --method dense      --config configs/dev.yaml --resume
!python -m rag_evidence.cli retrieve --method hybrid_rrf --config configs/dev.yaml --resume

In [ ]:
!python -m rag_evidence.cli generate --config configs/dev.yaml --resume

In [ ]:
%%bash
for m in citations embedding leave_one_out control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/dev.yaml --resume
done

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/dev.yaml
!python -m rag_evidence.cli report   --config configs/dev.yaml
!python -m rag_evidence.cli export   --config configs/dev.yaml
# download the dev zip NOW — don't wait until the end of the notebook, and don't rely on
# a later cell picking it up (a per-section download is what makes each section's result
# independently safe to take home)
import glob

from google.colab import files

dev_zips = sorted(glob.glob('results/export/results_dev_*.zip'))
print('dev export zips:', dev_zips)
if dev_zips:
    files.download(dev_zips[-1])

## Section 2 — LOCKED eval split (240 questions)

Longest section. Safe to run across several sessions — rerun from the status cell
after any disconnect.

In [ ]:
!python -m rag_evidence.cli status --config configs/full.yaml

In [ ]:
!python -m rag_evidence.cli retrieve --method dense      --config configs/full.yaml --resume
!python -m rag_evidence.cli retrieve --method hybrid_rrf --config configs/full.yaml --resume

In [ ]:
!python -m rag_evidence.cli generate --config configs/full.yaml --resume

In [ ]:
%%bash
for m in citations embedding leave_one_out control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/full.yaml --resume
done

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/full.yaml
!python -m rag_evidence.cli report   --config configs/full.yaml
!python -m rag_evidence.cli export   --config configs/full.yaml
# download EVERY export zip, not just the newest one: with both sections run in a single
# session there are two (dev + eval), and taking only zips[-1] silently drops dev's raw
# per-sample records — they exist only in their own zip.
import glob

from google.colab import files

zips = sorted(glob.glob('results/export/*.zip'))
print('export zips:', zips)
for z in zips:
    print('downloading', z)
    files.download(z)